In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('C:\\Users\\user\\Documents\\БНТУ\\Python\\путь до ML junior engineer\\Влад Бурмистров степик\\Ноутбуки\\08-Linear-Regression-Models\\Advertising.csv')

In [3]:
df.head()

,TV,radio,newspaper,sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [4]:
from sklearn.preprocessing import PolynomialFeatures

In [5]:
X = df.drop('sales', axis=1)
X

,TV,radio,newspaper
0,230.1,37.8,69.2
1,44.5,39.3,45.1
2,17.2,45.9,69.3
3,151.5,41.3,58.5
4,180.8,10.8,58.4
...,...,...,...
195,38.2,3.7,13.8
196,94.2,4.9,8.1
197,177.0,9.3,6.4
198,283.6,42.0,66.2


In [6]:
y = df['sales']

In [7]:
poly_converter = PolynomialFeatures(degree=5, include_bias=False)
poly_features = poly_converter.fit_transform(X)
len(poly_converter.get_feature_names())   # сделали 55 признаков умышленно  

55

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(poly_features, y, test_size=0.3, random_state=101)

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [11]:
model_over = LinearRegression()
model_over.fit(X_train, y_train)
over_predictions = model_over.predict(X_test)


mean_squared_error(y_test, over_predictions)   # мы видим сильное переобучение

6.63487319599914

In [12]:
train_over_predictions = model_over.predict(X_train)

mean_squared_error(y_train, train_over_predictions)

0.06296802050899301

In [13]:
from joblib import dump, load

In [14]:
loaded_lin = load('C:\\Users\\user\\Documents\\БНТУ\\Python\\путь до ML junior engineer\\Проекты\\Linear_reg\\final_sales_model.joblib')

In [15]:
# проверяем, насколько переобучилась наша модель с 19-ю признаками по сравнению с нормальной линейной моделью
experiment = [[149, 22, 12]]
pred_lin = loaded_lin.predict(experiment)
pred_lin   # предсказание нормальной модели

array([13.893032])

In [16]:
# сконвертим experiment для model_over:
experiment_feature = poly_converter.transform(experiment)
model_over.predict(experiment_feature)

array([14.28564302])

In [17]:
# поработаем с регуляризацией
# R2 - уменьшаем веса
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

In [18]:
ridge_model = RidgeCV()
standard_converter = StandardScaler()
standard_converter.fit_transform(X_train)
ridge_model.fit(standard_converter.transform(X_train), y_train)
ridge_model.alpha_   # наилучший alpha

0.1

In [19]:
ridge_model.predict(standard_converter.transform(X_test))


array([14.86836818, 19.24356123, 12.4480091 , 17.56125436,  7.95090194,
        7.19393969, 20.06691767, 17.22847386, 10.45992822, 19.63424067,
        9.33409374, 13.9711472 , 11.8778099 , 23.09876499, 18.98218108,
        9.51096741, 12.24626328,  9.24854976,  8.71289787, 21.15459319,
        7.27723476, 19.25634752, 26.93791975, 24.17385423,  9.17260613,
       11.54372785, 21.04131306,  9.47018034, 12.58803227,  8.18954231,
        8.80756279, 19.68529636, 10.87890231,  7.16719793, 17.99919076,
        9.40549577, 10.54498186,  9.71656938,  7.07385425, 10.89564647,
       11.32913867, 10.48537004, 11.39985639,  7.79750354, 11.81128488,
       10.6691667 ,  5.74164863, 15.32723772, 13.12131159, 22.4816505 ,
       10.58577407, 12.93307791, 14.65630383, 12.01483483, 11.43440498,
        5.64670996, 24.86785316,  9.28328132, 17.53224196, 15.52322016])

In [20]:
ridge_test_mse = mean_squared_error(y_test, ridge_model.predict(standard_converter.transform(X_test)))

In [21]:
ridge_train_mse = mean_squared_error(y_train, ridge_model.predict(standard_converter.transform(X_train)))

In [22]:
ridge_train_mse

0.18999994866109965

In [23]:
# ошибка на train была 0.06 - стала 0.19;

# ошибка на test была 6.6 - стала 0.44

In [24]:
# по итогу мы уменьшили степень переобученности, не меняя степени полинома, а просто корректируя веса, при это не обнуляя ни один из них